# 🧠 AntiHoax — Pelatihan Model IndoBERT + UMAP + GAT
**Pipeline:** IndoBERT Embedding → UMAP Dimensionality Reduction → Graph Attention Network (GAT)

> ⚠️ Pastikan **Runtime → Ubah jenis runtime → GPU (T4)** sebelum menjalankan.

---
### Cara Penggunaan:
1. Jalankan setiap cell secara berurutan
2. Upload file CSV dataset saat diminta
3. Setelah selesai, download `antihoax_model.pt` dan upload ke aplikasi

## 📦 Cell 1 — Instalasi Library

In [ ]:
# =============================================
# CELL 1: Install semua dependensi yang dibutuhkan
# =============================================
import subprocess
import sys

print('🔧 Menginstall PyTorch Geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric', '-q'], check=True)

print('🔧 Menginstall dependensi PyG...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv',
    '-f', 'https://data.pyg.org/whl/torch-2.3.0+cu121.html', '-q'
], check=True)

print('🔧 Menginstall transformers & UMAP...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'transformers', 'umap-learn', 'scikit-learn', '-q'
], check=True)

print('✅ Semua library berhasil diinstall!')

## 🔍 Cell 2 — Periksa GPU

In [ ]:
# =============================================
# CELL 2: Verifikasi GPU tersedia
# =============================================
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device yang digunakan: {device}')

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  GPU tidak tersedia! Pergi ke Runtime → Ubah jenis runtime → GPU')

## 📂 Cell 3 — Upload Dataset CSV

In [ ]:
# =============================================
# CELL 3: Upload file CSV dataset
# =============================================
from google.colab import files
import pandas as pd
import io

print('📂 Silakan upload file CSV dataset Anda...')
uploaded = files.upload()

# Ambil nama file pertama yang diupload
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'\n✅ File berhasil diupload: {filename}')
print(f'   Total baris: {len(df_raw)}')
print(f'   Kolom: {list(df_raw.columns)}')
print('\n🔍 Preview 5 baris pertama:')
df_raw.head()

## ⚙️ Cell 4 — Konfigurasi Parameter Training

In [ ]:
# =============================================
# CELL 4: Konfigurasi — UBAH SESUAI KEBUTUHAN
# =============================================

SETTINGS = {
    # --- Kolom Dataset ---
    # Ganti jika nama kolom berbeda
    'text_column':  None,  # None = otomatis deteksi
    'label_column': None,  # None = otomatis deteksi

    # --- IndoBERT ---
    'indo_batch_size': 16,       # Batch size untuk ekstraksi embedding (naikkan jika GPU besar)
    'indo_max_length': 128,      # Panjang token maksimum (max 512)
    'indobert_model': 'indobenchmark/indobert-base-p1',

    # --- UMAP ---
    'umap_n_components': 64,     # Dimensi output UMAP (rekomendasi: 32-128)
    'umap_n_neighbors': 15,      # Jumlah tetangga UMAP
    'umap_min_dist': 0.1,

    # --- KNN Graph ---
    'knn_k_neighbors': 10,       # Jumlah tetangga untuk membangun graph

    # --- GAT Training ---
    'gat_hidden_channels': 64,   # Ukuran hidden layer
    'gat_num_heads': 4,          # Jumlah attention heads
    'gat_dropout': 0.3,          # Dropout (rekomendasi: 0.2 - 0.4)
    'gat_learning_rate': 0.005,  # Learning rate
    'indo_epoch': 300,           # Maksimum epoch per fold
    'indo_fold': 5,              # Jumlah fold K-Fold
    'early_stop_patience': 30,   # Stop jika tidak ada perbaikan N epoch
}

print('✅ Konfigurasi berhasil dimuat:')
for k, v in SETTINGS.items():
    print(f'   {k}: {v}')

## 🧹 Cell 5 — Preprocessing Dataset

In [ ]:
# =============================================
# CELL 5: Preprocessing & Deteksi Kolom Otomatis
# =============================================
import numpy as np

df = df_raw.copy()

# 1. Hapus baris/kolom kosong total
df = df.dropna(axis=1, how='all').dropna(axis=0, how='all')

# 2. Deteksi header yang salah posisi
header_row_idx = -1
for idx, row in df.head(10).iterrows():
    row_str = ' '.join([str(x).lower() for x in row.values])
    if ('judul' in row_str or 'text' in row_str or 'narasi' in row_str) and \
       ('label' in row_str or 'target' in row_str or 'kelas' in row_str):
        header_row_idx = idx
        break

if header_row_idx != -1:
    df.columns = df.loc[header_row_idx]
    df = df.drop(index=df.index[:header_row_idx+1]).reset_index(drop=True)
    print(f'⚠️  Header diperbaiki dari baris {header_row_idx}')

# 3. Deteksi kolom teks & label
text_col  = SETTINGS.get('text_column')
label_col = SETTINGS.get('label_column')

if not text_col or not label_col:
    for col in df.columns:
        c = str(col).lower()
        if not text_col and any(k in c for k in ['text', 'judul', 'narasi', 'berita', 'konten']):
            text_col = col
        if not label_col and any(k in c for k in ['label', 'target', 'kelas', 'kategori']):
            label_col = col

if not text_col:
    # Fallback: kolom string terpanjang
    str_cols = [c for c in df.columns if df[c].dtype == object]
    if str_cols:
        text_col = max(str_cols, key=lambda c: df[c].str.len().mean())

if not label_col:
    # Fallback: kolom dengan 2-4 nilai unik
    for col in df.columns:
        if col != text_col and df[col].nunique() in [2, 3, 4]:
            label_col = col
            break

if not text_col or not label_col:
    text_col  = df.columns[0]
    label_col = df.columns[-1]

print(f'📌 Kolom teks  : "{text_col}"')
print(f'📌 Kolom label : "{label_col}"')

# 4. Bersihkan dan konversi
df = df.dropna(subset=[text_col, label_col])
texts     = [str(x) for x in df[text_col].tolist()]
raw_labels = df[label_col].tolist()
unique_vals = sorted(list(set(raw_labels)))

if any(isinstance(x, str) for x in raw_labels):
    labels = [unique_vals.index(v) for v in raw_labels]
else:
    if len(unique_vals) == 2:
        mn, mx = min(unique_vals), max(unique_vals)
        labels = [0 if v == mn else 1 for v in raw_labels]
    else:
        labels = [int(v) for v in raw_labels]

num_classes = len(unique_vals)

print(f'\n✅ Dataset siap:')
print(f'   Total data  : {len(texts)}')
print(f'   Jumlah kelas: {num_classes}')
print(f'   Mapping label: {dict(zip(range(len(unique_vals)), unique_vals))}')

# Tampilkan distribusi kelas
from collections import Counter
dist = Counter(labels)
print(f'   Distribusi  : {dict(dist)}')

## 🤖 Cell 6 — Ekstraksi Embedding IndoBERT

In [ ]:
# =============================================
# CELL 6: Ekstraksi Embedding dengan IndoBERT
# =============================================
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = SETTINGS['indobert_model']
BATCH_SIZE = SETTINGS['indo_batch_size']
MAX_LENGTH = SETTINGS['indo_max_length']

print(f'📥 Memuat model IndoBERT: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()
print(f'✅ IndoBERT berhasil dimuat ke {device}')

def get_embeddings(texts, batch_size=8):
    embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            outputs = bert_model(**encoded)
            # Gunakan CLS token sebagai representasi kalimat
            batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(batch_emb)

        batch_num = (i // batch_size) + 1
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f'   Batch {batch_num}/{total_batches} selesai ({len(embeddings)} teks terproses)')
    return embeddings

print(f'\n⚙️  Mengekstrak embedding dari {len(texts)} teks (batch={BATCH_SIZE})...')
embeddings = get_embeddings(texts, batch_size=BATCH_SIZE)
embeddings_np = np.array(embeddings)

print(f'\n✅ Embedding selesai!')
print(f'   Shape: {embeddings_np.shape}  (teks × dimensi IndoBERT)')

## 📉 Cell 7 — Reduksi Dimensi UMAP

In [ ]:
# =============================================
# CELL 7: Reduksi Dimensi dengan UMAP
# =============================================
import umap

N_COMPONENTS = SETTINGS['umap_n_components']
N_NEIGHBORS  = SETTINGS['umap_n_neighbors']
MIN_DIST     = SETTINGS['umap_min_dist']

# Pastikan n_neighbors tidak melebihi jumlah data
n_neighbors_safe = min(N_NEIGHBORS, len(embeddings_np) - 1)
# Pastikan n_components tidak melebihi embedding dimension
n_components_safe = min(N_COMPONENTS, embeddings_np.shape[1])

print(f'📉 Menjalankan UMAP: {embeddings_np.shape[1]}D → {n_components_safe}D')
print(f'   n_neighbors = {n_neighbors_safe}, min_dist = {MIN_DIST}')

reducer = umap.UMAP(
    n_components=n_components_safe,
    n_neighbors=n_neighbors_safe,
    min_dist=MIN_DIST,
    random_state=42,
    verbose=True
)
reduced_embs = reducer.fit_transform(embeddings_np)
reduced_embs = np.array(reduced_embs, dtype=np.float32)

print(f'\n✅ UMAP selesai!')
print(f'   Shape output: {reduced_embs.shape}')

## 🕸️ Cell 8 — Definisi Model GAT

In [ ]:
# =============================================
# CELL 8: Definisi Model Graph Attention Network (GAT)
# =============================================
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data

class ContentGraphGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=2, heads=4, dropout=0.3):
        super(ContentGraphGAT, self).__init__()
        self.dropout = dropout
        # Layer 1: GATConv
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        # Layer 2: GATConv — output = hidden * heads
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

print('✅ Arsitektur GAT berhasil didefinisikan')
print(f'   Layer 1: GATConv(in → hidden={SETTINGS["gat_hidden_channels"]}, heads={SETTINGS["gat_num_heads"]})')
print(f'   Layer 2: GATConv(hidden*heads → {num_classes}, heads=1)')
print(f'   Dropout: {SETTINGS["gat_dropout"]}')

## 🔗 Cell 9 — Bangun KNN Graph

In [ ]:
# =============================================
# CELL 9: Konstruksi Graph dengan KNN
# =============================================
from sklearn.neighbors import NearestNeighbors

def construct_graph(embeddings, k_neighbors=10):
    """
    Membuat sparse graph berdasarkan K-Nearest Neighbors.
    Lebih efisien memori dibanding fully connected graph.
    """
    n = len(embeddings)
    k = min(n - 1, k_neighbors)

    print(f'🔗 Membangun KNN graph: {n} node, k={k}...')
    knn = NearestNeighbors(n_neighbors=k, metric='cosine', n_jobs=-1)
    knn.fit(embeddings)
    distances, indices = knn.kneighbors(embeddings)

    edge_set = set()
    for i in range(n):
        for j in indices[i]:
            if i != j:
                edge_set.add((i, j))
                edge_set.add((j, i))  # Bidirectional

    if not edge_set:
        # Fallback: self-loops
        edge_set = {(i, i) for i in range(n)}

    edge_list = list(edge_set)
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    print(f'✅ Graph dibuat: {len(edge_set)} edges ({len(edge_set) // n:.1f} edge/node rata-rata)')
    return edge_index

K_NEIGHBORS = SETTINGS['knn_k_neighbors']
edge_index = construct_graph(reduced_embs, k_neighbors=K_NEIGHBORS)
print(f'   edge_index shape: {edge_index.shape}')

## 🏋️ Cell 10 — Training GAT dengan Stratified K-Fold

In [ ]:
# =============================================
# CELL 10: Training GAT dengan StratifiedKFold
# =============================================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Hyperparameter
N_SPLITS     = SETTINGS['indo_fold']
EPOCHS       = SETTINGS['indo_epoch']
HIDDEN       = SETTINGS['gat_hidden_channels']
HEADS        = SETTINGS['gat_num_heads']
DROPOUT      = SETTINGS['gat_dropout']
LR           = SETTINGS['gat_learning_rate']
PATIENCE     = SETTINGS['early_stop_patience']
IN_CHANNELS  = reduced_embs.shape[1]

# Tensor
x = torch.tensor(reduced_embs, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y).to(device)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

all_logs   = []
fold_best  = []
final_model = None

print(f'🏋️  Memulai training: {N_SPLITS}-Fold × maks {EPOCHS} Epoch')
print(f'   in_channels={IN_CHANNELS}, hidden={HIDDEN}, heads={HEADS}')
print(f'   lr={LR}, dropout={DROPOUT}, early_stop_patience={PATIENCE}')
print('=' * 70)

for fold, (train_idx, test_idx) in enumerate(skf.split(reduced_embs, labels)):
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask  = torch.zeros(data.num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True
    test_mask[test_idx]   = True

    data.train_mask = train_mask.to(device)
    data.test_mask  = test_mask.to(device)

    model = ContentGraphGAT(
        in_channels=IN_CHANNELS,
        hidden_channels=HIDDEN,
        out_channels=num_classes,
        heads=HEADS,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=20, factor=0.5, verbose=False
    )
    final_model = model

    best_loss       = float('inf')
    patience_counter = 0
    best_acc_fold   = 0.0

    print(f'\n📂 Fold {fold+1}/{N_SPLITS} — Train: {train_mask.sum().item()}, Test: {test_mask.sum().item()}')

    for epoch in range(1, EPOCHS + 1):
        # === Training ===
        model.train()
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        scheduler.step(loss)

        # === Early Stopping ===
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'   ⏹ Early stopping epoch {epoch} (patience={PATIENCE})')
                break

        # === Evaluasi ===
        model.eval()
        with torch.no_grad():
            pred       = model(data.x, data.edge_index).argmax(dim=1)
            test_preds = pred[data.test_mask].cpu().numpy()
            test_lbls  = data.y[data.test_mask].cpu().numpy()

        if len(test_lbls) > 0 and len(np.unique(test_lbls)) > 1:
            acc  = accuracy_score(test_lbls, test_preds)
            prec = precision_score(test_lbls, test_preds, average='macro', zero_division=0)
            rec  = recall_score(test_lbls, test_preds, average='macro', zero_division=0)
            f1   = f1_score(test_lbls, test_preds, average='macro', zero_division=0)
        else:
            acc = float(np.mean(test_preds == test_lbls)) if len(test_lbls) > 0 else 0
            prec = rec = f1 = acc

        best_acc_fold = max(best_acc_fold, acc)

        log = {
            'iterasi': f'F{fold+1} - E{epoch}',
            'loss': round(float(loss.item()), 4),
            'akurasi': round(float(acc), 4),
            'presisi': round(float(prec), 4),
            'recall': round(float(rec), 4),
            'f1': round(float(f1), 4),
        }
        all_logs.append(log)

        # Print setiap 10 epoch
        if epoch % 10 == 0 or epoch == 1:
            print(f'   Epoch {epoch:>3}/{EPOCHS} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')

    fold_best.append(best_acc_fold)
    print(f'   ✅ Fold {fold+1} selesai — Best Acc: {best_acc_fold:.4f}')

# Ringkasan
print('\n' + '=' * 70)
print(f'🏁 TRAINING SELESAI!')
print(f'   Akurasi per fold : {[round(a, 4) for a in fold_best]}')
print(f'   Rata-rata akurasi: {np.mean(fold_best):.4f} ± {np.std(fold_best):.4f}')

## 📊 Cell 11 — Visualisasi Hasil Training

In [ ]:
# =============================================
# CELL 11: Grafik Loss & Akurasi per Fold
# =============================================
import matplotlib.pyplot as plt
import pandas as pd

df_logs = pd.DataFrame(all_logs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AntiHoax GAT — Hasil Training', fontsize=14, fontweight='bold')

# Plot Loss
for fold_num in range(1, N_SPLITS + 1):
    fold_data = df_logs[df_logs['iterasi'].str.startswith(f'F{fold_num} -')]
    axes[0].plot(fold_data['loss'].values, label=f'Fold {fold_num}')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NLL Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Akurasi
for fold_num in range(1, N_SPLITS + 1):
    fold_data = df_logs[df_logs['iterasi'].str.startswith(f'F{fold_num} -')]
    axes[1].plot(fold_data['akurasi'].values, label=f'Fold {fold_num}')
axes[1].set_title('Akurasi Test per Fold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Akurasi')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabel metrik akhir
last_entry_per_fold = []
for fold_num in range(1, N_SPLITS + 1):
    fold_data = df_logs[df_logs['iterasi'].str.startswith(f'F{fold_num} -')]
    if not fold_data.empty:
        last_entry_per_fold.append(fold_data.iloc[-1])

summary_df = pd.DataFrame(last_entry_per_fold)[['iterasi', 'akurasi', 'presisi', 'recall', 'f1']]
print('\n📋 Metrik Akhir per Fold:')
print(summary_df.to_string(index=False))
print(f'\n   Mean Accuracy : {summary_df["akurasi"].mean():.4f}')
print(f'   Mean F1-Score : {summary_df["f1"].mean():.4f}')

## 💾 Cell 12 — Simpan & Download Model

In [ ]:
# =============================================
# CELL 12: Simpan model dan download
# =============================================
import json
import pickle

# 1. Simpan model GAT
torch.save(final_model.state_dict(), 'antihoax_gat_model.pt')
print('✅ Model GAT disimpan: antihoax_gat_model.pt')

# 2. Simpan UMAP reducer
with open('antihoax_umap_reducer.pkl', 'wb') as f:
    pickle.dump(reducer, f)
print('✅ UMAP reducer disimpan: antihoax_umap_reducer.pkl')

# 3. Simpan metadata (konfigurasi + label mapping)
metadata = {
    'settings': SETTINGS,
    'label_mapping': dict(zip(range(len(unique_vals)), [str(v) for v in unique_vals])),
    'num_classes': num_classes,
    'in_channels': IN_CHANNELS,
    'gat_config': {
        'hidden_channels': HIDDEN,
        'heads': HEADS,
        'dropout': DROPOUT,
        'out_channels': num_classes
    },
    'fold_accuracies': fold_best,
    'mean_accuracy': float(np.mean(fold_best)),
    'mean_f1': float(summary_df['f1'].mean()) if 'summary_df' in dir() else 0
}
with open('antihoax_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('✅ Metadata disimpan: antihoax_metadata.json')

# 4. Simpan log training
df_logs.to_csv('antihoax_training_logs.csv', index=False)
print('✅ Log training disimpan: antihoax_training_logs.csv')

# 5. Download semua file
from google.colab import files
print('\n📥 Mengunduh semua file...')
files.download('antihoax_gat_model.pt')
files.download('antihoax_umap_reducer.pkl')
files.download('antihoax_metadata.json')
files.download('antihoax_training_logs.csv')
files.download('training_plot.png')
print('\n🎉 Selesai! Semua file berhasil diunduh.')

## 🔎 Cell 13 — (Opsional) Uji Prediksi Manual

In [ ]:
# =============================================
# CELL 13: Uji model dengan teks baru (opsional)
# =============================================
TEKS_UJI = [
    # Ganti dengan teks yang ingin diuji
    "Pemerintah berencana memotong anggaran pendidikan sebesar 50 persen",
    "Dana BOS resmi dicairkan untuk seluruh sekolah dasar negeri",
]

def predict(texts_to_predict):
    # 1. Embed
    embs = get_embeddings(texts_to_predict, batch_size=4)
    embs_np = np.array(embs, dtype=np.float32)

    # 2. Reduce dengan UMAP (transform, bukan fit_transform)
    reduced = reducer.transform(embs_np).astype(np.float32)

    # 3. Bangun mini-graph
    n = len(reduced)
    if n > 1:
        ei = construct_graph(reduced, k_neighbors=min(n-1, K_NEIGHBORS))
    else:
        ei = torch.tensor([[0], [0]], dtype=torch.long)

    x_pred = torch.tensor(reduced, dtype=torch.float).to(device)
    ei_pred = ei.to(device)

    # 4. Prediksi
    final_model.eval()
    with torch.no_grad():
        out  = final_model(x_pred, ei_pred)
        prob = torch.exp(out)  # dari log_softmax ke probabilitas
        pred = out.argmax(dim=1).cpu().numpy()

    label_map = dict(zip(range(len(unique_vals)), unique_vals))
    results = []
    for i, text in enumerate(texts_to_predict):
        cls = label_map[pred[i]]
        confidence = float(prob[i].max().cpu())
        results.append({'teks': text[:80], 'prediksi': cls, 'confidence': f'{confidence:.2%}'})
    return results

results = predict(TEKS_UJI)
print('\n🔎 Hasil Prediksi:')
for r in results:
    print(f'  [{r["prediksi"]}] ({r["confidence"]}) — "{r["teks"]}"')